# Task 4: ArcFace


Restartable model notebook. It loads the saved Task 4 train split and preprocessing configuration from disk.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
import json
import math

import numpy as np
import optuna
import pandas as pd
import torch
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from pytorch_metric_learning import losses
from tqdm.auto import tqdm

from src.task4.config import (
    CONFIG_DIR,
    EARLY_STOPPING_PATIENCE,
    FINAL_EPOCHS,
    IMAGE_DIR,
    INPUT_SIZE,
    LABEL_COLUMN,
    LABEL_ID_COLUMN,
    MIN_CLASS_SIZE,
    MIN_DELTA,
    MODEL_DIR,
    N_TRIALS,
    SEED,
    SPLIT_DIR,
    TUNING_EPOCHS,
    VAL_FRACTION,
)
from src.task4.data import make_evaluation_loader, make_model_loaders
from src.task4.models import ResNet18Encoder
from src.task4.preprocessing import (
    cae_transform,
    metric_eval_transform,
    metric_preprocessing_config,
    metric_train_transform,
)
from src.task4.retrieval import (
    evaluate_retrieval,
    extract_embeddings,
    k_reciprocal_rerank,
    retrieval_metrics,
    search_cosine,
)
from src.task4.training import (
    AMP_ENABLED,
    DEVICE,
    EarlyStopping,
    cpu_state_dict,
    new_optimizer_and_scheduler,
    set_seed,
    train_one_epoch,
)
from src.task4.tuning import create_study, suggest_parameters


In [ ]:
set_seed(SEED)
print("Device:", DEVICE, "| Mixed precision:", AMP_ENABLED)

outer_train_df = pd.read_csv(SPLIT_DIR / "train.csv")
with (CONFIG_DIR / "articleType_gender_label_encoder.json").open(encoding="utf-8") as file:
    label_encoder_config = json.load(file)

classes = label_encoder_config["classes"]
label_to_index = label_encoder_config["label_to_index"]

if LABEL_COLUMN not in outer_train_df.columns:
    outer_train_df[LABEL_COLUMN] = (
        outer_train_df["articleType"].str.strip() + "__" + outer_train_df["gender"].str.strip()
    )
if LABEL_ID_COLUMN not in outer_train_df.columns:
    outer_train_df[LABEL_ID_COLUMN] = outer_train_df[LABEL_COLUMN].map(label_to_index).astype("int64")

outer_class_counts = outer_train_df[LABEL_COLUMN].value_counts()
eligible_labels = set(outer_class_counts[outer_class_counts >= MIN_CLASS_SIZE].index)
development_df = outer_train_df[outer_train_df[LABEL_COLUMN].isin(eligible_labels)].copy()

inner_stratify_columns = ["articleType", "gender"]
inner_stratify_features = pd.get_dummies(
    development_df[inner_stratify_columns].astype(str),
    prefix=inner_stratify_columns,
)
inner_splitter = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=VAL_FRACTION,
    random_state=SEED,
)
inner_train_idx, inner_val_idx = next(
    inner_splitter.split(development_df, inner_stratify_features)
)
train_df = development_df.iloc[inner_train_idx].copy()
val_df = development_df.iloc[inner_val_idx].copy()

overlap = set(train_df["id"]) & set(val_df["id"])
if overlap:
    raise ValueError(f"Inner split contains {len(overlap)} overlapping IDs")
if val_df[LABEL_ID_COLUMN].isna().any():
    raise ValueError("Inner validation contains an unmapped label")

gallery_parts = []
for _, group in train_df.groupby(LABEL_COLUMN, sort=True):
    sample_size = min(20, len(group))
    gallery_parts.append(group.sample(sample_size, random_state=SEED))

tuning_gallery_df = pd.concat(gallery_parts).sort_values("id").reset_index(drop=True)

print(f"Model train/validation: {len(train_df):,}/{len(val_df):,}")
print("Excluded rare rows:", len(outer_train_df) - len(development_df))
print("Tuning gallery rows:", len(tuning_gallery_df))


In [ ]:
def arcface_objective(trial: optuna.Trial):
    parameters = suggest_parameters(trial, "arcface")
    set_seed(SEED)
    model = ResNet18Encoder()
    model.arcface_loss = losses.ArcFaceLoss(
        num_classes=len(classes),
        embedding_size=512,
        margin=math.degrees(parameters.get("margin", 0.3)),
        scale=parameters.get("scale", 32),
    )
    loss_function = model.arcface_loss
    model, loss_function = model.to(DEVICE), loss_function.to(DEVICE)
    loaders = make_model_loaders(
        "arcface",
        train_df,
        tuning_gallery_df,
        val_df,
        cae_transform,
        metric_train_transform,
        metric_eval_transform,
        IMAGE_DIR,
    )
    optimizer, scheduler = new_optimizer_and_scheduler(model, parameters)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
    best_score = -math.inf
    for epoch in tqdm(range(1, TUNING_EPOCHS + 1), desc='arcface tuning'):
        train_one_epoch(
            "arcface", model, loss_function, loaders["train"], optimizer, scaler, epoch
        )
        score = evaluate_retrieval(model, loaders["gallery"], loaders["query"])[
            "mAP@10"
        ]
        scheduler.step(score)
        best_score = max(best_score, score)
        trial.report(score, step=epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return best_score

In [ ]:
arcface_study = create_study("task4_arcface")
arcface_study.optimize(arcface_objective, n_trials=N_TRIALS)
arcface_best_params = dict(arcface_study.best_params)
print("arcface", "best parameters:", arcface_best_params)

In [ ]:
arcface_history = []
arcface_best_epoch = 0
arcface_best_score = -math.inf

parameters = arcface_best_params
set_seed(SEED)
model = ResNet18Encoder()
model.arcface_loss = losses.ArcFaceLoss(
    num_classes=len(classes),
    embedding_size=512,
    margin=math.degrees(parameters.get("margin", 0.3)),
    scale=parameters.get("scale", 32),
)
loss_function = model.arcface_loss
model, loss_function = model.to(DEVICE), loss_function.to(DEVICE)
loaders = make_model_loaders(
    "arcface",
    train_df,
    tuning_gallery_df,
    val_df,
    cae_transform,
    metric_train_transform,
    metric_eval_transform,
    IMAGE_DIR,
)
optimizer, scheduler = new_optimizer_and_scheduler(model, parameters)
scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
stopper = EarlyStopping(EARLY_STOPPING_PATIENCE, MIN_DELTA)
best_state = None
for epoch in tqdm(range(1, FINAL_EPOCHS + 1), desc="arcface final"):
    train_loss = train_one_epoch(
        "arcface", model, loss_function, loaders["train"], optimizer, scaler, epoch
    )
    metrics = evaluate_retrieval(model, loaders["gallery"], loaders["query"])
    score = metrics["mAP@10"]
    scheduler.step(score)
    if stopper.update(score):
        best_state = cpu_state_dict(model)
        arcface_best_epoch = epoch
    arcface_history.append({"epoch": epoch, "train_loss": train_loss, **metrics})
    print("arcface", epoch, "loss:", round(train_loss, 4), "mAP@10:", round(score, 4))
    if stopper.should_stop:
        break
model.load_state_dict(best_state)
arcface_best_score = stopper.best_score
arcface_model = model
print("arcface", "best epoch:", arcface_best_epoch, "best mAP@10:", arcface_best_score)


In [ ]:
model_directory = MODEL_DIR / 'arcface'
model_directory.mkdir(parents=True, exist_ok=True)
checkpoint = {
    'model_name': 'arcface',
    'model_state_dict': cpu_state_dict(arcface_model),
    'model_config': {
        'backbone': 'resnet18', 'pretrained': False,
        'input_size': list(INPUT_SIZE), 'embedding_dimension': 512,
        'class_count': len(classes),
    },
    'best_params': arcface_best_params,
    'best_epoch': arcface_best_epoch,
    'best_val_map_at_10': arcface_best_score,
    'preprocessing_config': metric_preprocessing_config,
    'label_to_index': label_to_index,
    'gallery_scope': 'eligible_inner_training', 'random_seed': SEED,
}
torch.save(checkpoint, model_directory / 'best.pt')
gallery_loader = make_evaluation_loader(train_df, 'arcface', cae_transform, metric_eval_transform, IMAGE_DIR)
gallery_embeddings, gallery_ids, gallery_labels = extract_embeddings(arcface_model, gallery_loader)
np.save(model_directory / 'gallery_embeddings.npy', gallery_embeddings)
np.save(model_directory / 'gallery_ids.npy', gallery_ids)
query_loader = make_evaluation_loader(val_df, 'arcface', cae_transform, metric_eval_transform, IMAGE_DIR)
query_embeddings, _, query_labels = extract_embeddings(arcface_model, query_loader)
_, base_indices, index = search_cosine(query_embeddings, gallery_embeddings)
reranked_indices = k_reciprocal_rerank(query_embeddings, gallery_embeddings, index)
print('arcface', 'base:', retrieval_metrics(base_indices, query_labels, gallery_labels))
print('arcface', 'reranked:', retrieval_metrics(reranked_indices, query_labels, gallery_labels))
print('Saved:', model_directory / 'best.pt')


Observation note: record best validation mAP@10, convergence behavior, and artifact paths here.
